# 03 — Model 1: TF-IDF + Cosine Similarity
**ResumeAI Project** | Baseline keyword matching model

## 3.1 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              accuracy_score, confusion_matrix, classification_report)
from collections import Counter
import re, warnings
warnings.filterwarnings('ignore')

print("All libraries loaded!")

## 3.2 Sample Dataset

In [ ]:
# Representative resume-JD pairs with ground truth match labels
# Label 1 = good match (score >= 60), Label 0 = poor match
data = [
    {
        'resume': "python developer machine learning tensorflow keras deep learning neural networks data science sklearn pandas numpy aws docker kubernetes ci cd git agile scrum rest api flask fastapi sql nosql postgresql mongodb",
        'jd':     "machine learning engineer python tensorflow keras deep learning aws docker kubernetes data science numpy pandas sql postgresql rest api agile",
        'label':  1, 'true_score': 85
    },
    {
        'resume': "react javascript frontend developer html css tailwind nodejs expressjs mongodb rest api git agile ui ux responsive design webpack vite redux",
        'jd':     "react developer javascript html css nodejs rest api git agile frontend mongodb responsive",
        'label':  1, 'true_score': 82
    },
    {
        'resume': "java spring boot microservices docker kubernetes aws jenkins ci cd postgresql redis kafka rest api agile scrum maven gradle junit",
        'jd':     "java spring boot microservices docker kubernetes aws postgresql kafka rest api ci cd agile backend developer",
        'label':  1, 'true_score': 88
    },
    {
        'resume': "graphic designer photoshop illustrator figma adobe xd branding logo design print media creative direction color theory typography",
        'jd':     "python developer machine learning artificial intelligence data science tensorflow deep learning aws cloud",
        'label':  0, 'true_score': 12
    },
    {
        'resume': "data analyst excel tableau power bi sql reporting visualization business intelligence stakeholder communication presentations",
        'jd':     "data scientist python machine learning tensorflow deep learning neural networks aws kubernetes docker",
        'label':  0, 'true_score': 28
    },
    {
        'resume': "devops engineer aws azure gcp docker kubernetes jenkins terraform ansible ci cd linux bash python monitoring grafana prometheus",
        'jd':     "cloud engineer aws kubernetes docker terraform ci cd devops linux python monitoring",
        'label':  1, 'true_score': 87
    },
    {
        'resume': "android developer java kotlin android studio firebase rest api sqlite mvvm clean architecture unit testing espresso",
        'jd':     "react developer javascript html css nodejs rest api git agile frontend responsive design",
        'label':  0, 'true_score': 22
    },
    {
        'resume': "python django rest framework postgresql celery redis docker git linux agile tdd unit testing backend developer api development",
        'jd':     "backend developer python django rest api postgresql redis docker celery linux agile tdd",
        'label':  1, 'true_score': 91
    },
    {
        'resume': "project manager pmp agile scrum stakeholder management budget planning risk management team leadership communication ms project jira",
        'jd':     "software engineer python javascript react nodejs aws docker kubernetes backend frontend full stack",
        'label':  0, 'true_score': 15
    },
    {
        'resume': "fullstack developer react nodejs express mongodb python flask postgresql docker aws git ci cd rest api agile javascript typescript",
        'jd':     "full stack developer react nodejs mongodb postgresql docker aws rest api javascript typescript agile ci cd",
        'label':  1, 'true_score': 90
    },
]

df = pd.DataFrame(data)
print(f"Dataset size: {len(df)} resume-JD pairs")
print(f"Match distribution: {df['label'].value_counts().to_dict()}")
print(f"\nSample pair:")
print(f"  Resume (first 80 chars): {df.iloc[0]['resume'][:80]}...")
print(f"  JD (first 80 chars):     {df.iloc[0]['jd'][:80]}...")
print(f"  Label: {df.iloc[0]['label']} | True Score: {df.iloc[0]['true_score']}")

## 3.3 TF-IDF Vectorization

In [ ]:
# Initialize TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2),      # unigrams and bigrams
    min_df=1,
    max_df=0.95,
    sublinear_tf=True,
)

# Fit on all texts
all_texts = list(df['resume']) + list(df['jd'])
vectorizer.fit(all_texts)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)} terms")
print(f"\nTop 20 terms in vocabulary:")
feature_names = vectorizer.get_feature_names_out()
print(list(feature_names[:20]))

In [ ]:
# Compute TF-IDF scores and cosine similarity for each pair
results = []

for idx, row in df.iterrows():
    resume_vec = vectorizer.transform([row['resume']])
    jd_vec     = vectorizer.transform([row['jd']])

    similarity = cosine_similarity(resume_vec, jd_vec)[0][0]
    ats_score  = round(similarity * 100, 1)
    prediction = 1 if ats_score >= 50 else 0

    results.append({
        'pair_id':    idx + 1,
        'tfidf_similarity': round(similarity, 4),
        'tfidf_score':      ats_score,
        'true_score':       row['true_score'],
        'true_label':       row['label'],
        'predicted_label':  prediction,
        'correct':          prediction == row['label'],
    })

results_df = pd.DataFrame(results)
print("TF-IDF Scores for each resume-JD pair:")
print(results_df.to_string(index=False))

## 3.4 Evaluation Metrics

In [ ]:
y_true = results_df['true_label']
y_pred = results_df['predicted_label']

precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)
accuracy  = accuracy_score(y_true, y_pred)

print("TF-IDF MODEL EVALUATION METRICS")
print("=" * 40)
print(f"  Precision : {precision:.4f} ({precision*100:.1f}%)")
print(f"  Recall    : {recall:.4f} ({recall*100:.1f}%)")
print(f"  F1-Score  : {f1:.4f} ({f1*100:.1f}%)")
print(f"  Accuracy  : {accuracy:.4f} ({accuracy*100:.1f}%)")
print()
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=['No Match', 'Match']))

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('TF-IDF Model Results', fontsize=13, fontweight='bold')

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted: No Match','Predicted: Match'],
            yticklabels=['Actual: No Match','Actual: Match'],
            ax=axes[0], cbar=False, annot_kws={'size':14,'weight':'bold'})
axes[0].set_title('Confusion Matrix')

# Predicted vs True scores
axes[1].scatter(results_df['true_score'], results_df['tfidf_score'],
                c=['#22c55e' if c else '#ef4444' for c in results_df['correct']],
                s=100, alpha=0.85, edgecolors='white', linewidth=0.5)
axes[1].plot([0,100],[0,100],'--', color='gray', alpha=0.5, label='Perfect prediction')
axes[1].set_xlabel('True ATS Score')
axes[1].set_ylabel('TF-IDF Predicted Score')
axes[1].set_title('True Score vs TF-IDF Score')
green_patch = plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='#22c55e', markersize=10, label='Correct')
red_patch   = plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='#ef4444', markersize=10, label='Incorrect')
axes[1].legend(handles=[green_patch, red_patch])
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('03_tfidf_model.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.5 Keyword Analysis — What TF-IDF Extracts

In [ ]:
def get_top_tfidf_keywords(text, vectorizer, top_n=10):
    """Get top TF-IDF weighted keywords for a text"""
    vec = vectorizer.transform([text])
    scores = zip(vectorizer.get_feature_names_out(), vec.toarray()[0])
    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return [(word, round(score, 4)) for word, score in sorted_scores[:top_n] if score > 0]

for i, row in df.head(3).iterrows():
    print(f"\nPair {i+1} (Match={row['label']}):")
    print("  Resume top keywords:")
    for kw, sc in get_top_tfidf_keywords(row['resume'], vectorizer):
        print(f"    {kw}: {sc}")
    print("  JD top keywords:")
    for kw, sc in get_top_tfidf_keywords(row['jd'], vectorizer):
        print(f"    {kw}: {sc}")

## 3.6 TF-IDF Model Summary

In [ ]:
print("TF-IDF MODEL SUMMARY")
print("=" * 45)
print(f"  Algorithm   : TF-IDF + Cosine Similarity")
print(f"  Vocabulary  : {len(vectorizer.vocabulary_)} terms")
print(f"  N-grams     : (1,2) — unigrams + bigrams")
print(f"  Accuracy    : {accuracy*100:.1f}%")
print(f"  Precision   : {precision*100:.1f}%")
print(f"  Recall      : {recall*100:.1f}%")
print(f"  F1-Score    : {f1*100:.1f}%")
print()
print("  Strengths:")
print("    + Very fast (< 0.1 seconds per pair)")
print("    + No GPU required")
print("    + Fully interpretable — can see exact keyword weights")
print("    + Good at exact keyword matching")
print()
print("  Limitations:")
print("    - Cannot understand semantic meaning")
print("    - 'developer' and 'engineer' treated as different terms")
print("    - Sensitive to vocabulary differences")
print("    - Ignores word order and context")
print()
print("  Conclusion: Good baseline model but limited by lack of semantic understanding.")
print("  Used in ResumeAI as a fast pre-processing step before LLM analysis.")